# Ekstraksi Fitur Tiga Dimensi
Tiga dimensi kemiripan, tiap fitur menghasilkan skor 0–1:

| Dimensi | Metode | Input | Dasar hukum (Penjelasan Ps. 21) |
|---|---|---|---|
| Tekstual | Jaro-Winkler | `nama_base` | persamaan tulisan |
| Fonetik | Double Metaphone | `nama_phon` | persamaan bunyi ucapan |
| Semantik | FastText pretrained cc.id.300 | `nama_base` | persamaan makna|

**Catatan penting yang notebook ini terapkan:**
1. Semantik memakai **pretrained cc.id.300**, bukan self-trained. Self-trained terbukti tidak diskriminatif pada nama merek pendek (skor ~1.0 untuk semua). Pretrained satu-satunya yang bisa menangkap kemiripan makna nyata.
2. Gold cases dikategorikan: **bermakna** (string berbeda → evaluation utama), **trivial** (string identik → dilaporkan terpisah), **dikecualikan** (kontradiktif & reputasi_dominan).


## 0. Setup

In [ ]:
!pip install rapidfuzz metaphone gensim fasttext-wheel -q
import os, re, numpy as np, pandas as pd
from rapidfuzz.distance import JaroWinkler
from metaphone import doublemetaphone
print("Setup selesai.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 14.0 MB/s eta 0:00:00
Setup selesai.


## 1. Muat data preprocessed & modul preprocessing
Upload `haystack_preprocessed_fonID.csv`, `haystack_didaftar_fonID.csv`, `gold_cases_preprocessed_fonID.csv`, dan `preprocessing_merek.py` (dari Langkah B).


In [ ]:
for f in ["haystack_preprocessed_fonID.csv","haystack_didaftar_fonID.csv",
          "gold_cases_preprocessed_fonID.csv","preprocessing_merek.py"]:
    if not os.path.exists(f):
        from google.colab import files
        print("Upload file dari Langkah B (haystack, gold, preprocessing_merek.py):")
        files.upload()
        break

hay      = pd.read_csv("haystack_preprocessed_fonID.csv", dtype=str).fillna("")
hay_daftar = pd.read_csv("haystack_didaftar_fonID.csv", dtype=str).fillna("")
gold     = pd.read_csv("gold_cases_preprocessed_fonID.csv", dtype=str).fillna("")
print("Haystack:", hay.shape, "| Didaftar:", hay_daftar.shape, "| Gold:", gold.shape)

Upload file dari Langkah B (haystack, gold, preprocessing_merek.py):


Saving preprocessing_merek.py to preprocessing_merek.py
Haystack: (13514, 11) | Didaftar: (7650, 11) | Gold: (95, 18)


## 2. Kategorisasi gold cases
- **bermakna**: string berbeda → evaluation utama.
- **trivial_identik**: string sama & label mirip/identik → positif trivial, dilaporkan terpisah.
- **dikecualikan_kontradiktif**: string sama tapi label `tidak_mirip` → mustahil dipelajari model teks, dikeluarkan.
- **dikecualikan_reputasi**: label `reputasi_dominan` → dikeluarkan dari metrik utama.


In [ ]:
gold["identik_str"] = (gold["merek_a_base"]==gold["merek_b_base"]) & (gold["merek_a_base"]!="")

def kategori(r):
    lab = r["label_usulan"]
    if lab == "reputasi_dominan":            return "dikecualikan_reputasi"
    if r["identik_str"] and lab=="tidak_mirip": return "dikecualikan_kontradiktif"
    if r["identik_str"]:                     return "trivial_identik"
    return "bermakna"

gold["kategori_eval"] = gold.apply(kategori, axis=1)
print(gold["kategori_eval"].value_counts().to_string())

gold_eval    = gold[gold["kategori_eval"]=="bermakna"].reset_index(drop=True)
gold_trivial = gold[gold["kategori_eval"]=="trivial_identik"].reset_index(drop=True)
gold_excl    = gold[gold["kategori_eval"].str.startswith("dikecualikan")].reset_index(drop=True)

print(f"\nEvaluation utama (bermakna): {len(gold_eval)} "
      f"-> {gold_eval['label_usulan'].value_counts().to_dict()}")
print("CATATAN: kelas timpang (mirip >> tidak_mirip). Utamakan recall & lapor per-kelas.")
gold.to_csv("gold_cases_categorized.csv", index=False, encoding="utf-8-sig")
gold_eval.to_csv("gold_eval_bermakna.csv", index=False, encoding="utf-8-sig")

kategori_eval
bermakna                     55
trivial_identik              32
dikecualikan_reputasi         4
dikecualikan_kontradiktif     4

Evaluation utama (bermakna): 55 -> {'mirip': 47, 'tidak_mirip': 8}
CATATAN: kelas timpang (mirip >> tidak_mirip). Utamakan recall & lapor per-kelas.


## 3. Fitur 1 — Tekstual (Jaro-Winkler pada `nama_base`)

In [ ]:
def jw_sim(a, b):
    if not a or not b: return 0.0
    return JaroWinkler.normalized_similarity(a, b)

# sanity
print("KOJIE-SAN vs KOJIE-SAN DREAM WHITE:", round(jw_sim("kojie san","kojie san dream white"),3))
print("MS GLOW vs PS GLOW:", round(jw_sim("ms glow","ps glow"),3))

KOJIE-SAN vs KOJIE-SAN DREAM WHITE: 0.886
MS GLOW vs PS GLOW: 0.905


## 4. Fitur 2 — Fonetik (Double Metaphone pada `nama_phon`)
Tiap token di-encode Double Metaphone (primary & secondary), lalu kode dibandingkan dengan Jaro-Winkler agar menghasilkan skor bertahap (bukan cocok/tidak biner). Diambil kombinasi primary/secondary terbaik.


In [ ]:
def _dm_codes(text):
    toks = text.split()
    prim = "".join(doublemetaphone(t)[0] for t in toks)
    sec  = "".join((doublemetaphone(t)[1] or doublemetaphone(t)[0]) for t in toks)
    return prim, sec

def phon_sim(a, b):
    if not a or not b: return 0.0
    pa, sa = _dm_codes(a); pb, sb = _dm_codes(b)
    cands = [JaroWinkler.normalized_similarity(x, y)
             for x in (pa, sa) for y in (pb, sb) if x and y]
    return max(cands) if cands else 0.0

print("DAIMARU vs DIAMARU:", round(phon_sim("daimaru","diamaru"),3))
print("MS GLOW vs PS GLOW:", round(phon_sim("ms glow","ps glow"),3))

DAIMARU vs DIAMARU: 1.0
MS GLOW vs PS GLOW: 0.933


## 5. Fitur 3 — Semantik (pretrained cc.id.300, di-cache ke Drive)
**Strategi Colab gratis:** unduh cc.id.300.bin sekali (~4GB) → reduksi ke 100 dimensi → simpan ke Google Drive → sesi berikutnya muat versi kecil dari Drive. Ada fallback aman bila memori tidak cukup.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/skripsi_merek"
os.makedirs(DRIVE_DIR, exist_ok=True)
REDUCED_PATH = f"{DRIVE_DIR}/cc.id.100.bin"
SEM_DIM = 100

Mounted at /content/drive


In [ ]:
import fasttext
try:
    import fasttext.util
except Exception:
    pass

sem_model, SEM_SOURCE = None, None
try:
    if os.path.exists(REDUCED_PATH):
        print("Memuat model semantik tereduksi dari Drive...")
        sem_model = fasttext.load_model(REDUCED_PATH)
        SEM_SOURCE = "pretrained_cc.id.300_reduced100 (dari Drive)"
    else:
        print("Mengunduh cc.id.300.bin (~4GB, HANYA sekali)...")
        fasttext.util.download_model("id", if_exists="ignore")   # -> cc.id.300.bin
        print("Memuat model penuh (butuh RAM besar)...")
        sem_model = fasttext.load_model("cc.id.300.bin")
        print(f"Reduksi dimensi 300 -> {SEM_DIM} ...")
        fasttext.util.reduce_model(sem_model, SEM_DIM)
        sem_model.save_model(REDUCED_PATH)
        print("Model tereduksi disimpan ke Drive:", REDUCED_PATH)
        SEM_SOURCE = "pretrained_cc.id.300_reduced100 (baru)"
    print("Sumber semantik:", SEM_SOURCE)
except Exception as e:
    print("GAGAL memuat pretrained (kemungkinan memori kurang):", repr(e))
    print(">> Fallback self-trained. PERINGATAN: biasanya NON-diskriminatif pada nama merek.")
    from gensim.models import FastText as GensimFT
    corpus = [n.split() for n in hay["nama_base"] if n.strip()]
    _ft = GensimFT(vector_size=100, window=3, min_count=1, sg=1, epochs=20, seed=42)
    _ft.build_vocab(corpus); _ft.train(corpus, total_examples=len(corpus), epochs=20)
    sem_model = ("gensim", _ft)
    SEM_SOURCE = "self_trained_fallback"

Memuat model semantik tereduksi dari Drive...
Sumber semantik: pretrained_cc.id.300_reduced100 (dari Drive)


In [ ]:
def _sem_vec(text):
    if sem_model is None or not str(text).strip():
        return None
    if isinstance(sem_model, tuple) and sem_model[0]=="gensim":
        m = sem_model[1]; toks=[t for t in text.split() if t]
        if not toks: return None
        return np.mean([m.wv[t] for t in toks], axis=0)
    return sem_model.get_sentence_vector(str(text).replace("\n"," "))

def sem_sim(a, b):
    va, vb = _sem_vec(a), _sem_vec(b)
    if va is None or vb is None: return 0.0
    na, nb = np.linalg.norm(va), np.linalg.norm(vb)
    return float(np.dot(va, vb)/(na*nb)) if na and nb else 0.0

print("Uji semantik (harusnya BERBEDA jika pretrained bekerja):")
print("  rose vs mawar     :", round(sem_sim("rose","mawar"),3))
print("  harimau vs macan  :", round(sem_sim("harimau","macan"),3))
print("  starbucks vs meja :", round(sem_sim("starbucks","meja"),3))

Uji semantik (harusnya BERBEDA jika pretrained bekerja):
  rose vs mawar     : 0.579
  harimau vs macan  : 0.803
  starbucks vs meja : 0.429


## 6. Sanity check — apakah tiap fitur mendiskriminasi pada set bermakna?
Skor `mirip` harus lebih tinggi dari `tidak_mirip`. Kalau semantik tetap datar, itu temuan (dokumentasikan; grid search akan memberinya bobot rendah).


In [ ]:
def tiga_fitur(row):
    return pd.Series({
        "jw":   jw_sim(row["merek_a_base"], row["merek_b_base"]),
        "phon": phon_sim(row["merek_a_phon"], row["merek_b_phon"]),
        "sem":  sem_sim(row["merek_a_base"], row["merek_b_base"]),
    })

feat = gold_eval.join(gold_eval.apply(tiga_fitur, axis=1))
print("=== Rata-rata skor fitur per label (set bermakna) ===")
print(feat.groupby("label_usulan")[["jw","phon","sem"]].mean().round(3).to_string())
print("\nSumber semantik:", SEM_SOURCE)

# selisih mirip - tidak_mirip sebagai indikator daya pisah
m = feat.groupby("label_usulan")[["jw","phon","sem"]].mean()
if {"mirip","tidak_mirip"}.issubset(m.index):
    print("\nDaya pisah (mirip - tidak_mirip):")
    print((m.loc["mirip"] - m.loc["tidak_mirip"]).round(3).to_string())
feat.to_csv("gold_eval_dengan_fitur.csv", index=False, encoding="utf-8-sig")

=== Rata-rata skor fitur per label (set bermakna) ===
                 jw   phon    sem
label_usulan                     
mirip         0.779  0.806  0.681
tidak_mirip   0.624  0.617  0.562

Sumber semantik: pretrained_cc.id.300_reduced100 (dari Drive)

Daya pisah (mirip - tidak_mirip):
jw      0.155
phon    0.189
sem     0.120


## 7. Pra-hitung embedding haystack (untuk retrieval Langkah D–F)
Menghitung sekali vektor semantik seluruh haystack agar retrieval cepat.


In [ ]:
names = hay["nama_base"].tolist()
vecs = []
for n in names:
    v = _sem_vec(n)
    vecs.append(v if v is not None else np.zeros(SEM_DIM, dtype="float32"))
emb = np.vstack(vecs).astype("float32")
np.save("haystack_emb.npy", emb)
hay[["id","nama_merek","nama_base","status_permohonan"]].to_csv(
    "haystack_emb_index.csv", index=False, encoding="utf-8-sig")
print("Embedding haystack:", emb.shape, "-> haystack_emb.npy + haystack_emb_index.csv")

Embedding haystack: (13514, 100) -> haystack_emb.npy + haystack_emb_index.csv


## 8. Simpan modul fitur (untuk Langkah D–F)
JW & fonetik bersifat deterministik → disimpan sebagai fungsi. Semantik bergantung pada model yang dimuat, jadi di notebook berikutnya muat ulang model dari Drive (cepat).


In [ ]:
modul = '''from rapidfuzz.distance import JaroWinkler
from metaphone import doublemetaphone

def jw_sim(a, b):
    if not a or not b: return 0.0
    return JaroWinkler.normalized_similarity(a, b)

def _dm_codes(text):
    toks = text.split()
    prim = "".join(doublemetaphone(t)[0] for t in toks)
    sec  = "".join((doublemetaphone(t)[1] or doublemetaphone(t)[0]) for t in toks)
    return prim, sec

def phon_sim(a, b):
    if not a or not b: return 0.0
    pa, sa = _dm_codes(a); pb, sb = _dm_codes(b)
    cands = [JaroWinkler.normalized_similarity(x, y)
             for x in (pa, sa) for y in (pb, sb) if x and y]
    return max(cands) if cands else 0.0
'''
with open("fitur_merek.py","w",encoding="utf-8") as f: f.write(modul)
import importlib, fitur_merek; importlib.reload(fitur_merek)
print("fitur_merek.py tersimpan.")

fitur_merek.py tersimpan.


## 9. Unduh
Model semantik tereduksi sudah tersimpan permanen di Drive (`skripsi_merek/cc.id.100.bin`) — sesi berikutnya muat dari sana, tak perlu unduh 4GB lagi.

**Lanjut ke Langkah D–F:** bangun struktur retrieval (query vs haystack), grid search bobot (w·JW + w·phon + w·sem) & threshold, evaluasi (MAP/MRR/Recall@k + klasifikasi), bandingkan hybrid vs single-method, dan jalankan ablasi (fonetik-ID on/off; semantik pretrained vs tanpa).


In [ ]:
from google.colab import files
for f in ["gold_cases_categorized.csv","gold_eval_bermakna.csv","gold_eval_dengan_fitur.csv",
          "haystack_emb.npy","haystack_emb_index.csv","fitur_merek.py"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>